## Observação: 
 
#### Com os 54.840 pontos de dado este notebook demora em média 40 minutos para rodar na minha máquina.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords")
import numpy as np
import pandas as pd
import json
import glob

#gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim.models import TfidfModel

#spacy
import spacy
from nltk.corpus import stopwords

#vis
import pyLDAvis
import pyLDAvis.gensim_models

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

C:\ProgramData\Anaconda3\lib\site-packages\sklearn\feature_extraction\image.py:167: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dtype=np.int):
C:\ProgramData\Anaconda3\lib\site-packages\sklearn\linear_model\least_angle.py:30: DeprecationWarning: `np.float` is a deprecated alias for the builtin `float`. To silence this warning, use `float` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.float64` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.o

In [2]:
df = pd.read_csv('database/journals.csv')
df = df.iloc[np.where(~df['Abstract'].isna())]
df = df.iloc[np.where(~df['Publication Year'].isna())]

# Descritivas

In [3]:
print(len(df))

53038


In [4]:
df['Source Title'].value_counts().sort_index()

ARTIFICIAL INTELLIGENCE REVIEW                                     1378
COMPUTATIONAL LINGUISTICS                                           574
EXPERT SYSTEMS WITH APPLICATIONS                                  15809
FOUNDATIONS AND TRENDS IN MACHINE LEARNING                           22
IEEE COMPUTATIONAL INTELLIGENCE MAGAZINE                            267
IEEE TRANSACTIONS ON KNOWLEDGE AND DATA ENGINEERING                4313
IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARNING SYSTEMS          3292
IEEE TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE     5609
IEEE Transactions on Neural Networks and Learning Systems             5
INTERNATIONAL JOURNAL OF COMPUTER VISION                           2365
INTERNATIONAL JOURNAL OF INTELLIGENT SYSTEMS                       2318
INTERNATIONAL JOURNAL OF ROBOTICS RESEARCH                         2195
JOURNAL OF ARTIFICIAL INTELLIGENCE RESEARCH                        1363
JOURNAL OF MACHINE LEARNING RESEARCH                            

In [5]:
df['Publication Year'].value_counts().sort_index()

1990.0      10
1991.0     406
1992.0     529
1993.0     600
1994.0     600
1995.0     683
1996.0     716
1997.0     719
1998.0     760
1999.0     713
2000.0     684
2001.0     738
2002.0     877
2003.0     935
2004.0    1089
2005.0    1101
2006.0    1178
2007.0    1305
2008.0    1663
2009.0    2467
2010.0    2258
2011.0    2819
2012.0    2883
2013.0    2192
2014.0    2259
2015.0    2391
2016.0    2293
2017.0    2305
2018.0    2695
2019.0    2668
2020.0    3093
2021.0    4302
2022.0    3107
Name: Publication Year, dtype: int64

### Pontos de dado com a data disponível variando mensalmente

In [6]:
sum(df['Publication Date'].value_counts()[:12])

37396

### Pontos de dado com a data disponível variando diariamente

In [7]:
sum(df['Publication Date'].value_counts()[12:])

12116

In [8]:
data = list(df['Abstract'])

In [9]:
stopwords = stopwords.words("english")

In [10]:
def lemmatization(texts, allowed_postags=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    return [" ".join([token.lemma_ for token in nlp(text) if token.pos_ in allowed_postags]) for text in texts]

lemmatized_texts = lemmatization(data)

In [11]:
def gen_words(texts):
    return [gensim.utils.simple_preprocess(text, deacc=True) for text in texts]

data_words = gen_words(lemmatized_texts)

In [26]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=50)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=50)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return [bigram[doc] for doc in texts]

def make_trigrams(texts):
    return [trigram[bigram[doc]] for doc in texts]

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams) 

In [ ]:
#TF-IDF
id2word = corpora.Dictionary(data_bigrams_trigrams)
texts = data_bigrams_trigrams
corpus = [id2word.doc2bow(text) for text in texts]

tfidf = TfidfModel(corpus, id2word=id2word)

low_value=0.03
words = []
words_missing_in_tfidf = []

for i in range(0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] #reinitialize to be safe. You can skip this.
    tfidf_ids = [id for id, value in tfidf[bow]]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words + words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # The words with tf-idf socre 0 will be missing

    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]       
    corpus[i] = new_bow

In [ ]:
words

In [15]:
# id2word = corpora.Dictionary(data_words)
# corpus = [id2word.doc2bow(text) for text in data_words]
# word = id2word[[0][:1][0]]

In [16]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=30,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [17]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word, mds="mmds", R=30)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster  Freq
topic                                           
0     -0.001364  0.000501       1        1   NaN
1      0.000483 -0.001125       2        1   NaN
2      0.001703  0.000441       3        1   NaN
3      0.000512  0.000575       4        1   NaN
4      0.001360 -0.000582       5        1   NaN
5     -0.000098 -0.001194       6        1   NaN
6     -0.000892  0.001473       7        1   NaN
7      0.001380  0.000211       8        1   NaN
8     -0.001154 -0.000928       9        1   NaN
9     -0.001760 -0.000218      10        1   NaN
10    -0.000366  0.000916      11        1   NaN
11    -0.000324 -0.001548      12        1   NaN
12    -0.001406 -0.001297      13        1   NaN
13     0.000971  0.000007      14        1   NaN
14     0.000330  0.001682      15        1   NaN
15     0.001711 -0.000783      16        1   NaN
16     0.000922 -0.000883      17        1   NaN
17    -0.001629  0.000842      18        1   NaN
18    -0.000551 -0.000466      19        1   NaN
19     0.000644  0.001232      20        1   NaN
20     0.000917  0.001489      21        1   NaN
21    -0.000530  0.001288      22        1   NaN
22    -0.000791  0.001746      23        1   NaN
23     0.001082 -0.001328      24        1   NaN
24     0.001558  0.000924      25        1   NaN
25    -0.000873 -0.001329      26        1   NaN
26     0.000285 -0.001818      27        1   NaN
27    -0.001246 -0.000248      28        1   NaN
28    -0.000143 -0.000015      29        1   NaN
29    -0.000732  0.000435      30        1   NaN, topic_info=                            Term  Freq  Total Category  logprob  loglift
0                           also   0.0    0.0  Default     30.0     30.0
1                       approach   0.0    0.0  Default     29.0     29.0
2                      attention   0.0    0.0  Default     28.0     28.0
3                           base   0.0    0.0  Default     27.0     27.0
4                      benchmark   0.0    0.0  Default     26.0     26.0
5                          block   0.0    0.0  Default     25.0     25.0
6                        cascade   0.0    0.0  Default     24.0     24.0
7                        channel   0.0    0.0  Default     23.0     23.0
8                   characterize   0.0    0.0  Default     22.0     22.0
9                        collect   0.0    0.0  Default     21.0     21.0
10                   consequence   0.0    0.0  Default     20.0     20.0
11                       consist   0.0    0.0  Default     19.0     19.0
12                    continuous   0.0    0.0  Default     18.0     18.0
13                   convolution   0.0    0.0  Default     17.0     17.0
14  convolutional_neural_network   0.0    0.0  Default     16.0     16.0
15                         datum   0.0    0.0  Default     15.0     15.0
16                          deep   0.0    0.0  Default     14.0     14.0
17                   demonstrate   0.0    0.0  Default     13.0     13.0
18                        easily   0.0    0.0  Default     12.0     12.0
19                    eeg_signal   0.0    0.0  Default     11.0     11.0
20                     effective   0.0    0.0  Default     10.0     10.0
21                     electrode   0.0    0.0  Default      9.0      9.0
22             epileptic_seizure   0.0    0.0  Default      8.0      8.0
23                         exist   0.0    0.0  Default      7.0      7.0
24                  experimental   0.0    0.0  Default      6.0      6.0
25                        expert   0.0    0.0  Default      5.0      5.0
26                           few   0.0    0.0  Default      4.0      4.0
27                           fix   0.0    0.0  Default      3.0      3.0
28                gate_recurrent   0.0    0.0  Default      2.0      2.0
29                      identify   0.0    0.0  Default      1.0      1.0, token_table=Empty DataFrame
Columns: [Topic, Freq, Term]
Index: [], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab':

In [18]:
corpus

[]